In [ ]:
from dataset.cell_motion import CellMotion
try:
    from dataset.cell_motion_rollout import CellMotionRollout
except ImportError:
    from dataset.cell_motion import CellMotionRollout
from model.simulator import Simulator
import torch
from torch_geometric.loader import DataLoader
import torch_geometric.transforms as T
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os
import csv
from datetime import datetime
from typing import Optional

# 数据集路径
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# 训练参数
batch_size = 16  
sequence_length = 20  
max_epochs = 1500  

# 预训练模型加载
load_model_path = None 

# 网络参数
hidden_size=256
message_passing_num = 15  
node_input_size = sequence_length * 2 + 1  
edge_input_size = 3  
learning_rate = 1e-03 

# 学习率调度参数
lr_patience = 50       
lr_factor = 0.5        
min_lr = 1e-6          

# 训练控制
print_batch = 10  
save_epoch = 5  

# ============================================

# 设置计算设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 初始化模拟器模型
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = os.path.join(f'checkpoint', timestamp)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

simulator = Simulator(
    message_passing_num=message_passing_num,
    node_input_size=node_input_size,
    edge_input_size=edge_input_size,
    device=device,
    hidden_size=hidden_size,
    model_dir=save_dir 
)

# 加载预训练权重
if load_model_path and os.path.exists(load_model_path):
    print(f'Loading model weights from: {load_model_path}')
    try:
        if hasattr(simulator, 'load_checkpoint'):
             simulator.load_checkpoint(load_model_path)
        else:
            checkpoint = torch.load(load_model_path, map_location=device)
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                simulator.load_state_dict(checkpoint['model_state_dict'])
            else:
                simulator.load_state_dict(checkpoint)
        print('Model weights loaded successfully.')
    except Exception as e:
        print(f'Error loading model: {e}')

# 使用Adam优化器
optimizer = torch.optim.Adam(simulator.parameters(), lr=learning_rate)
print('Optimizer initialized')

# 初始化学习率调度器
scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=lr_factor, 
    patience=lr_patience, 
    min_lr=min_lr,
    verbose=True
)
print('LR Scheduler initialized')

# 定义图变换流程
transformer = T.Compose([
    T.Cartesian(norm=False),
    T.Distance(norm=False)
])

def compute_velocity_features(graph_x, sequence_len):
    """
    辅助函数：将绝对位置特征转换为速度（差分）特征。
    graph.x 的结构是 [batch_size, sequence_length * 2 + 1]
    其中前 sequence_length * 2 是 (x_t-k, y_t-k, ..., x_t, y_t)
    """
    # 剥离时间特征（最后一个维度）
    time_feature = graph_x[:, -1:]
    pos_features = graph_x[:, :-1]
    
    # 重塑为 [num_nodes, sequence_length, 2]
    # 注意：这里需要根据您 Dataset 中具体的 flatten 顺序来调整
    num_nodes = pos_features.size(0)
    pos_seq = pos_features.view(num_nodes, sequence_len, 2)
    
    # 计算速度 v_t = p_t - p_{t-1}
    # 对于第一个时间步，由于没有前一个时间步，我们可以补0或者复制第二个时间步的速度
    # 这里采用计算相邻差分：结果形状 [num_nodes, sequence_len - 1, 2]
    velocities = pos_seq[:, 1:, :] - pos_seq[:, :-1, :]
    # 为了保持输入维度一致（或者您可以根据需要填充），通常需要处理掉减少的一个时间步
    # 方法1：填充第一帧速度为0
    zeros = torch.zeros(num_nodes, 1, 2, device=graph_x.device)
    velocities_padded = torch.cat([zeros, velocities], dim=1)
    
    # 展平回 [num_nodes, sequence_length * 2]
    velocities_flat = velocities_padded.view(num_nodes, -1)
    
    # 拼回时间特征
    new_features = torch.cat([velocities_flat, time_feature], dim=1)
    
    return new_features

def train(model: Simulator, dataloader, optimizer, current_epoch, *,
          max_grad_norm: float = 1.0,
          max_grad_value: Optional[float] = None):
    """训练函数"""
    epoch_loss = 0.0
    batch_count = 0
    model.train()
        
    for batch_index, graph in enumerate(dataloader):
        graph = transformer(graph)
        graph = graph.to(device)
        
        # -----------------------------------------------------------
        # 修改部分开始：输入特征转换为速度
        # -----------------------------------------------------------
        
        # 原始节点特征 (绝对位置序列)
        original_x = graph.x 
        
        # 转换为速度特征输入给模型
        # 注意：这里假设您希望模型基于历史速度来推理
        velocity_input = compute_velocity_features(original_x, sequence_length)
        
        # 临时替换图的输入特征，传入模型
        # 如果您的 Simulator 内部不仅用 graph.x 还需要 graph.pos，请确保逻辑一致
        graph.x = velocity_input
        
        # 模型预测的是 下一步的速度 (predicted_velocity)
        predicted_velocity = model.model(graph)

        # -----------------------------------------------------------
        # 修改部分：计算目标速度 (Target Velocity)
        # -----------------------------------------------------------
        
        target_positions = graph.y
        current_positions = graph.pos
        
        # 目标速度 = 下一步真实位置 - 当前位置
        # 这本质上就是之前的 displacement，但物理意义现在是 velocity * dt (假设dt=1)
        target_velocity = target_positions - current_positions

        # 归一化 (依然很重要，只是现在归一化的是速度分布)
        target_velocity_normalized = model._output_normalizer(
            target_velocity,
            model.training 
        )
        
        # 计算 Loss (预测速度 vs 真实速度)
        mse_loss = torch.mean((predicted_velocity - target_velocity_normalized) ** 2)
        loss = mse_loss 
        
        optimizer.zero_grad()
        loss.backward()

        if max_grad_norm is not None and max_grad_norm > 0:
            params_with_grad = [p for group in optimizer.param_groups for p in group['params'] if p.grad is not None]
            if len(params_with_grad) > 0:
                torch.nn.utils.clip_grad_norm_(params_with_grad, max_grad_norm)
        
        optimizer.step()

        epoch_loss += loss.item()
        batch_count += 1
        
        # (可选) 恢复 graph.x 以防后续还需要用原始数据（虽在这里不需要）
        graph.x = original_x

    avg_loss = epoch_loss / batch_count if batch_count > 0 else 0.0
    
    if current_epoch % save_epoch == 0 and current_epoch > 0:
        check_point_name = f'epoch={current_epoch}.pth'
        check_point_dir = os.path.join(save_dir, check_point_name)
        model.save_checkpoint(check_point_dir)

    return avg_loss

def validate(model: Simulator, dataloader):
    """验证函数"""
    model.eval()
    epoch_loss = 0.0
    batch_count = 0
    
    with torch.no_grad():
        for graph in dataloader:
            graph = transformer(graph)
            graph = graph.to(device)
            
            target_positions = graph.y
            current_positions = graph.pos
            
            # -----------------------------------------------------------
            # 修改部分：验证时同样转换输入为速度，并进行积分验证
            # -----------------------------------------------------------
            
            # 1. 构建速度输入
            velocity_input = compute_velocity_features(graph.x, sequence_length)
            graph.x = velocity_input
            
            # 2. 预测速度
            predicted_velocity_normalized = model.model(graph)
            
            # 3. 反归一化得到物理空间的速度 (Predicted V)
            predicted_velocity = model._output_normalizer.inverse(predicted_velocity_normalized)
            
            # 4. 积分得到预测的下一步位置 (Integrate: P_next = P_curr + V_pred)
            predicted_next_position = current_positions + predicted_velocity
            
            # 5. 计算损失
            # 这里您可以选择计算 "速度误差" 或 "积分后的位置误差"
            # 既然目标是位移预测，计算积分后的位置误差可能更直观，
            # 但为了和训练Loss保持量级一致以便观察，这里展示 速度误差（即位移增量误差）。
            
            target_velocity = target_positions - current_positions
            
            # 为了计算Loss，我们通常在归一化空间计算，或者都反归一化。
            # 这里保持与训练一致：在归一化空间计算 MSE
            target_velocity_normalized = model._output_normalizer(target_velocity, False)
            
            mse_loss = torch.mean((predicted_velocity_normalized - target_velocity_normalized) ** 2)
            
            epoch_loss += mse_loss.item()
            batch_count += 1
            
    avg_loss = epoch_loss / batch_count if batch_count > 0 else 0.0
    return avg_loss

if __name__ == '__main__':
    print('=' * 60)
    print('Cell Motion Prediction Training (Velocity Based)')
    print('=' * 60)
    
    #  加载训练集
    dataset_train = CellMotion(
        dataset_dir=dataset_dir,
        split='train_e0n300',
        max_epochs=max_epochs,
        sequence_length=sequence_length
    )
    train_loader = DataLoader(
        dataset=dataset_train,
        batch_size=batch_size,
        num_workers=0
    )
    
    #  加载验证集
    print("Loading Validation Set...")
    dataset_valid = CellMotionRollout(
        dataset_dir=dataset_dir, 
        sequence_length=sequence_length, 
        split='valid_e0n400'
    )
    valid_loader = DataLoader(
        dataset=dataset_valid, 
        batch_size=batch_size, 
        num_workers=0
    )

    # 训练日志配置
    log_dir = save_dir
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f'training_log_{timestamp}.csv')
    
    best_epoch_dir = os.path.join(save_dir, 'best_epoch')
    os.makedirs(best_epoch_dir, exist_ok=True)
    
    print(f'\nTraining log will be saved to: {log_file}')
    
    with open(log_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Epoch', 'Train_Loss', 'Valid_Loss', 'Timestamp', 'Is_Best', 'Learning_Rate'])
    
    best_valid_loss = float('inf')
    best_epoch_num = 0
    
    print('\nStarting training...\n')
    try:
        for epoch in range(max_epochs):  
            current_epoch = epoch + 1
            
            train_loss = train(simulator, train_loader, optimizer, current_epoch)
            valid_loss = validate(simulator, valid_loader)
            
            before_lr = optimizer.param_groups[0]['lr']
            scheduler.step(valid_loss)
            current_lr = optimizer.param_groups[0]['lr']
            
            is_best = valid_loss < best_valid_loss
            if is_best:
                best_valid_loss = valid_loss
                best_epoch_num = current_epoch
                print(f'\nNew best valid loss: {best_valid_loss:.4e}')
                
                best_checkpoint_path = os.path.join(
                    best_epoch_dir, 
                    f'best_checkpoint.pth'
                )
                simulator.save_checkpoint(best_checkpoint_path)
            
            print(f'Epoch {current_epoch} | Train Loss: {train_loss:.4e} | Valid Loss: {valid_loss:.4e} | LR: {current_lr:.2e}')

            with open(log_file, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                writer.writerow([current_epoch, train_loss, valid_loss, current_time, is_best, current_lr])
            
    except KeyboardInterrupt:
        print('\n\nTraining interrupted by user')
        simulator.save_checkpoint()
        print('Final checkpoint saved')
    
    print('\nTraining completed!')
    print(f'Training log saved to: {log_file}')
    print(f'Best valid loss: {best_valid_loss:.4e} at epoch {best_epoch_num}')

Using device: cuda


In [1]:
from dataset.cell_motion import CellMotion
try:
    from dataset.cell_motion_rollout import CellMotionRollout
except ImportError:
    from dataset.cell_motion import CellMotionRollout
from model.simulator import Simulator
import torch
from torch_geometric.loader import DataLoader
# 引入 PyG 的 DataParallel
from torch_geometric.nn import DataParallel 
import torch_geometric.transforms as T
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os
import csv
from datetime import datetime
from typing import Optional

# 数据集路径
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# 训练参数
# 注意：多卡训练时，batch_size 通常指的是总 batch size。
# 如果你有4张卡，每张卡实际分到的 batch 是 batch_size / 4
batch_size = 32  
sequence_length = 20  
max_epochs = 1500  

# 预训练模型加载
load_model_path = None 

# 网络参数
hidden_size=256
message_passing_num = 15  
node_input_size = sequence_length * 2 + 1  
edge_input_size = 3  
learning_rate = 1e-03 

# 学习率调度参数
lr_patience = 50       
lr_factor = 0.5        
min_lr = 1e-6          

# 训练控制
print_batch = 10  
save_epoch = 5  

# ============================================

# 1. 设置可见的 CUDA 设备
# 这样程序看到的 cuda:0 其实是物理上的 cuda:0, cuda:1 是物理上的 cuda:1，以此类推
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

# 设置主计算设备
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device_ids = [0, 1, 2, 3] # 指定使用的设备 ID 列表
print(f'Using device: {device}')
print(f'Using GPU IDs: {device_ids}')

# 初始化模拟器模型
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = os.path.join(f'checkpoint', timestamp)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 实例化基础模型
# 注意：Simulator 内部如果有 .to(device) 操作，最好指向主设备(cuda:0)
base_simulator = Simulator(
    message_passing_num=message_passing_num,
    node_input_size=node_input_size,
    edge_input_size=edge_input_size,
    device=device,
    hidden_size=hidden_size,
    model_dir=save_dir 
)

# 加载预训练权重 (在包装 DataParallel 之前加载)
if load_model_path and os.path.exists(load_model_path):
    print(f'Loading model weights from: {load_model_path}')
    try:
        if hasattr(base_simulator, 'load_checkpoint'):
             base_simulator.load_checkpoint(load_model_path)
        else:
            checkpoint = torch.load(load_model_path, map_location=device)
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                base_simulator.load_state_dict(checkpoint['model_state_dict'])
            else:
                base_simulator.load_state_dict(checkpoint)
        print('Model weights loaded successfully.')
    except Exception as e:
        print(f'Error loading model: {e}')

# 2. 包装模型以支持多卡并行
# 注意：这里使用的是 torch_geometric.nn.DataParallel，专门用于处理图数据
if torch.cuda.device_count() > 1:
    print(f"Let's use {torch.cuda.device_count()} GPUs!")
    # DataParallel 会自动将 input mini-batch 切分并散布到各个 GPU
    simulator = DataParallel(base_simulator, device_ids=device_ids)
    simulator = simulator.to(device)
else:
    simulator = base_simulator.to(device)

# 使用Adam优化器
optimizer = torch.optim.Adam(simulator.parameters(), lr=learning_rate)
print('Optimizer initialized')

# 初始化学习率调度器
scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=lr_factor, 
    patience=lr_patience, 
    min_lr=min_lr,
    verbose=True
)
print('LR Scheduler initialized')

# 定义图变换流程
transformer = T.Compose([
    T.Cartesian(norm=False),
    T.Distance(norm=False)
])

def compute_velocity_features(graph_x, sequence_len):
    """
    辅助函数：将绝对位置特征转换为速度（差分）特征。
    """
    time_feature = graph_x[:, -1:]
    pos_features = graph_x[:, :-1]
    
    num_nodes = pos_features.size(0)
    pos_seq = pos_features.view(num_nodes, sequence_len, 2)
    
    velocities = pos_seq[:, 1:, :] - pos_seq[:, :-1, :]
    zeros = torch.zeros(num_nodes, 1, 2, device=graph_x.device)
    velocities_padded = torch.cat([zeros, velocities], dim=1)
    
    velocities_flat = velocities_padded.view(num_nodes, -1)
    
    new_features = torch.cat([velocities_flat, time_feature], dim=1)
    
    return new_features

def train(model, dataloader, optimizer, current_epoch, *,
          max_grad_norm: float = 1.0,
          max_grad_value: Optional[float] = None):
    """训练函数"""
    epoch_loss = 0.0
    batch_count = 0
    model.train()
        
    for batch_index, graph in enumerate(dataloader):
        graph = transformer(graph)
        # DataParallel 会处理设备分发，但主图对象通常需要先移动到主设备
        # 或者在使用 DataParallel 时，通常输入是一个 list of data objects��PyG DataParallel 会处理
        # 但这里的 dataloader 输出已经是 Batch 对象了。
        # PyG 的 DataParallel 接受 Batch 对象并在内部 scatter。
        # 所以我们只需要把数据放到 CPU 或主 GPU 上即可，DataParallel forward 时会处理。
        # 为了保险起见，我们通常手动 .to(device) 到主设备
        graph = graph.to(device)
        
        # -----------------------------------------------------------
        # 修改部分开始：输入特征转换为速度
        # -----------------------------------------------------------
        
        original_x = graph.x 
        velocity_input = compute_velocity_features(original_x, sequence_length)
        graph.x = velocity_input
        
        # 模型预测
        # DataParallel 的 forward 调用会自动分发 graph 到各个卡
        # 注意：你需要确保 model.model (Simulator内的网络) 的输出与这里处理逻辑一致
        # 如果 model 是 DataParallel 包装后的，直接调用 model(graph) 
        # Simulator 类可能需要修改 forward 方法来返回 predicted_velocity
        # 这里假设 Simulator.model 是实际的 GNN，且 Simulator 继承自 nn.Module
        
        # !重要!: 由于 Simulator 包装了 DataParallel，现在访问内部属性需要用 model.module
        if isinstance(model, DataParallel):
             # 如果 model 本身是 Simulator 实例被包装了
             # DataParallel forward 会调用 Simulator.forward
             # 我们需要确保 Simulator.forward 返回的是 predicted_velocity
             predicted_velocity = model(graph)
             
             # 访问 output_normalizer 需要通过 module
             normalizer = model.module._output_normalizer
             is_training = model.module.training
        else:
             predicted_velocity = model.model(graph) # 旧逻辑，如果 Simulator 没有 forward 方法
             normalizer = model._output_normalizer
             is_training = model.training

        # -----------------------------------------------------------
        # 修改部分：计算目标速度 (Target Velocity)
        # -----------------------------------------------------------
        
        target_positions = graph.y
        current_positions = graph.pos
        
        target_velocity = target_positions - current_positions

        # 归一化 
        target_velocity_normalized = normalizer(
            target_velocity,
            is_training 
        )
        
        # DataParallel 在多卡上计算时，输出可能会被 gather 到主卡
        # 此时 predicted_velocity 和 target_velocity_normalized 的形状应该匹配
        
        mse_loss = torch.mean((predicted_velocity - target_velocity_normalized) ** 2)
        loss = mse_loss 
        
        optimizer.zero_grad()
        loss.backward()

        if max_grad_norm is not None and max_grad_norm > 0:
            params_with_grad = [p for group in optimizer.param_groups for p in group['params'] if p.grad is not None]
            if len(params_with_grad) > 0:
                torch.nn.utils.clip_grad_norm_(params_with_grad, max_grad_norm)
        
        optimizer.step()

        epoch_loss += loss.item()
        batch_count += 1
        
        # 恢复 graph.x
        graph.x = original_x

    avg_loss = epoch_loss / batch_count if batch_count > 0 else 0.0
    
    # 保存模型时，如果是 DataParallel，建议保存 model.module
    if current_epoch % save_epoch == 0 and current_epoch > 0:
        check_point_name = f'epoch={current_epoch}.pth'
        check_point_dir = os.path.join(save_dir, check_point_name)
        
        model_to_save = model.module if isinstance(model, DataParallel) else model
        # 假设 Simulator 有 save_checkpoint 方法
        if hasattr(model_to_save, 'save_checkpoint'):
            model_to_save.save_checkpoint(check_point_dir)
        else:
            torch.save(model_to_save.state_dict(), check_point_dir)

    return avg_loss

def validate(model, dataloader):
    """验证函数"""
    model.eval()
    epoch_loss = 0.0
    batch_count = 0
    
    with torch.no_grad():
        for graph in dataloader:
            graph = transformer(graph)
            graph = graph.to(device) # 到主设备
            
            target_positions = graph.y
            current_positions = graph.pos
            
            velocity_input = compute_velocity_features(graph.x, sequence_length)
            graph.x = velocity_input
            
            # 预测
            if isinstance(model, DataParallel):
                predicted_velocity_normalized = model(graph)
                normalizer = model.module._output_normalizer
            else:
                predicted_velocity_normalized = model.model(graph)
                normalizer = model._output_normalizer
            
            # 反归一化
            predicted_velocity = normalizer.inverse(predicted_velocity_normalized)
            
            # 积分得到预测的下一步位置
            predicted_next_position = current_positions + predicted_velocity
            
            target_velocity = target_positions - current_positions
            
            target_velocity_normalized = normalizer(target_velocity, False)
            
            mse_loss = torch.mean((predicted_velocity_normalized - target_velocity_normalized) ** 2)
            
            epoch_loss += mse_loss.item()
            batch_count += 1
            
    avg_loss = epoch_loss / batch_count if batch_count > 0 else 0.0
    return avg_loss

if __name__ == '__main__':
    print('=' * 60)
    print('Cell Motion Prediction Training (Velocity Based) - Multi GPU')
    print('=' * 60)
    
    #  加载训练集
    dataset_train = CellMotion(
        dataset_dir=dataset_dir,
        split='train_e0n300',
        max_epochs=max_epochs,
        sequence_length=sequence_length
    )
    # PyG 的 DataListLoader 在 DataParallel 下可能更高效，但 DataLoader 配合 PyG DataParallel 也是标准的
    train_loader = DataLoader(
        dataset=dataset_train,
        batch_size=batch_size,
        num_workers=4, # 增加 worker 数量以加快数据加载
    )
    
    #  加载验证集
    print("Loading Validation Set...")
    dataset_valid = CellMotionRollout(
        dataset_dir=dataset_dir, 
        sequence_length=sequence_length, 
        split='valid_e0n400'
    )
    valid_loader = DataLoader(
        dataset=dataset_valid, 
        batch_size=batch_size, 
        num_workers=4,
    )

    # ... (后续日志部分代码基本不变) ...
    # 训练日志配置
    log_dir = save_dir
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f'training_log_{timestamp}.csv')
    
    best_epoch_dir = os.path.join(save_dir, 'best_epoch')
    os.makedirs(best_epoch_dir, exist_ok=True)
    
    print(f'\nTraining log will be saved to: {log_file}')
    
    with open(log_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Epoch', 'Train_Loss', 'Valid_Loss', 'Timestamp', 'Is_Best', 'Learning_Rate'])
    
    best_valid_loss = float('inf')
    best_epoch_num = 0
    
    print('\nStarting training...\n')
    try:
        for epoch in range(max_epochs):  
            current_epoch = epoch + 1
            
            train_loss = train(simulator, train_loader, optimizer, current_epoch)
            valid_loss = validate(simulator, valid_loader)
            
            before_lr = optimizer.param_groups[0]['lr']
            scheduler.step(valid_loss)
            current_lr = optimizer.param_groups[0]['lr']
            
            is_best = valid_loss < best_valid_loss
            if is_best:
                best_valid_loss = valid_loss
                best_epoch_num = current_epoch
                print(f'\nNew best valid loss: {best_valid_loss:.4e}')
                
                best_checkpoint_path = os.path.join(
                    best_epoch_dir, 
                    f'best_checkpoint.pth')
                
                # 保存时注意解包 DataParallel
                model_to_save = simulator.module if isinstance(simulator, DataParallel) else simulator
                if hasattr(model_to_save, 'save_checkpoint'):
                    model_to_save.save_checkpoint(best_checkpoint_path)
                else:
                    torch.save(model_to_save.state_dict(), best_checkpoint_path)
            
            print(f'Epoch {current_epoch} | Train Loss: {train_loss:.4e} | Valid Loss: {valid_loss:.4e} | LR: {current_lr:.2e}')

            with open(log_file, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                writer.writerow([current_epoch, train_loss, valid_loss, current_time, is_best, current_lr])
            
    except KeyboardInterrupt:
        print('\n\nTraining interrupted by user')
        model_to_save = simulator.module if isinstance(simulator, DataParallel) else simulator
        if hasattr(model_to_save, 'save_checkpoint'):
            model_to_save.save_checkpoint()
        print('Final checkpoint saved')
    
    print('\nTraining completed!')
    print(f'Training log saved to: {log_file}')
    print(f'Best valid loss: {best_valid_loss:.4e} at epoch {best_epoch_num}')

Using device: cuda:0
Using GPU IDs: [0, 1, 2, 3]
Simulator model initialized with hidden_size=256
Let's use 4 GPUs!
Optimizer initialized
LR Scheduler initialized
Cell Motion Prediction Training (Velocity Based) - Multi GPU
CellMotion Dataset /data/home/zhaoyj/wansheng/meshgn/meshGraphNets/dataset/train_e0n300.h5 Initialized (sequence_length=20)
Loading Validation Set...

Training log will be saved to: checkpoint/20260204_161001/training_log_20260204_161001.csv

Starting training...



AttributeError: 'tuple' object has no attribute 'num_nodes'